In [3]:
!pip install transformers datasets torch scikit-learn evaluate accelerate

  Using cached transformers-5.0.0-py3-none-any.whl.metadata (37 kB)
  Using cached datasets-4.5.0-py3-none-any.whl.metadata (19 kB)
  Using cached torch-2.10.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached filelock-3.20.3-py3-none-any.whl.metadata (2.1 kB)
  Using cached huggingface_hub-1.3.4-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer_slim-0.21.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached hf_xet-1.2.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.9 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import datasets

dataset = "chengxuphd/liar2"
dataset = datasets.load_dataset(dataset)

statement_train, y_train = dataset["train"]["statement"], dataset["train"]["label"]
statement_val, y_val = dataset["validation"]["statement"], dataset["validation"]["label"]
statement_test, y_test = dataset["test"]["statement"], dataset["test"]["label"]

model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)


class StatementDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
                    text,
                    add_special_tokens=True,
                    max_length=self.max_len,
                    padding='max_length',
                    truncation=True,
                    return_attention_mask=True,
                    return_tensors='pt',
                )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


train_dataset = StatementDataset(statement_train, y_train, tokenizer)
val_dataset = StatementDataset(statement_val, y_val, tokenizer)
test_dataset = StatementDataset(statement_test, y_test, tokenizer)


def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# 5. Chargement du modèle
# num_labels doit correspondre au nombre de classes dans vos données (ex: 2 pour binaire)
num_labels = len(set(y_train)) 
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# 6. Configuration de l'entraînement
training_args = TrainingArguments(
    output_dir='./results',          # Dossier de sortie
    num_train_epochs=3,              # Nombre d'époques
    per_device_train_batch_size=16,  # Taille du batch d'entraînement
    per_device_eval_batch_size=32,   # Taille du batch d'évaluation
    warmup_steps=500,                # Steps de chauffe pour le learning rate
    weight_decay=0.01,               # Régularisation
    logging_dir='./logs',            # Dossier de logs
    logging_steps=10,
    eval_strategy="epoch",    # Evaluer à la fin de chaque époque
    save_strategy="epoch",           # Sauvegarder à la fin de chaque époque
    load_best_model_at_end=True,     # Charger le meilleur modèle à la fin
)

# 7. Création du Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# 8. Lancer l'entraînement
trainer.train()

# 9. Evaluation finale sur le test set
results = trainer.evaluate(test_dataset)
print("Résultats sur le test set :", results)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 892.59it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

Epoch,Training Loss,Validation Loss
